# 02 - Retrieval experiments

Turns three design claims into measurements:

1. dense and lexical retrieval fail on different queries
2. fusing them beats either alone
3. cross-encoder reranking improves the ordering further

Uses known-item retrieval: a document title is the query, the document's
own chunks are the relevant set. No human judgements, so it reproduces
exactly.

In [1]:
import sys
from pathlib import Path

CAPSTONE = Path(r"C:\Users\modij\Downloads\MUJ-DS-23FE10CDS00439\capstone")
sys.path.insert(0, str(CAPSTONE / "src"))

from dotenv import load_dotenv
load_dotenv(CAPSTONE / ".env")

import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

In [2]:
from branched_rag.pipeline import BranchedRAGPipeline
from branched_rag.evaluation import evaluate_retrieval

pipeline = BranchedRAGPipeline.load(CAPSTONE / 'config.yaml')
print('generation backend:', pipeline.llm.description)
pipeline.store.stats()

C:\Users\modij\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


generation backend: groq:openai/gpt-oss-120b


,branch,label,chunks,share_of_corpus,top_k,dense_weight,sparse_weight
0,reviews,Customer Reviews,4436,0.267,6,0.7,0.3
1,news,Market News,2040,0.123,6,0.6,0.4
2,financials,Financial Reporting,10165,0.611,5,0.5,0.5


## Where dense and lexical retrieval disagree

A figure-heavy query is the clearest case.

In [3]:
index = pipeline.store['financials']
query = 'operating profit rose to EUR 47.5 million'

dense = index.dense.search(pipeline.embedder.encode_one(query), 5)
sparse = index.sparse.search(query, 5)

print('DENSE top 3')
for row, score in dense[:3]:
    print(f'  {score:.3f}  {index.chunks.iloc[row].text[:95]}')
print()
print('LEXICAL top 3')
for row, score in sparse[:3]:
    print(f'  {score:.3f}  {index.chunks.iloc[row].text[:95]}')

DENSE top 3
  0.576  It forecast 2018 non-IFRS operating profit of 7.3 to 7.5 billion euros, adding that the impleme
  0.576  Excluding the impact of all gains on sales, total fees and other operating income of $38.5 mill
  0.555  Gross profit margin for the fourth quarter decreased 340 basis points as compared to the prior 

LEXICAL top 3
  18.810  Jan 30 (Reuters) - Comdirect Bank AG: * 2017 PRE-TAX PROFIT: EUR94.9M * DIVIDEND TO BE PROPOSED
  17.512  Jan 17 (Reuters) - PRO KAPITAL GRUPP AS : * POSITIVE PROFIT WARNING FOR 2018 FROM AS PRO KAPITA
  16.991  Looking at individual stocks, Britain's Capita tumbled to the bottom of the European benchmark 


In [4]:
overlap = {r for r, _ in dense} & {r for r, _ in sparse}
print(f'rows in both top-5 lists: {len(overlap)} of 5')
print('The two retrievers largely disagree, which is why fusing them helps.')

rows in both top-5 lists: 0 of 5
The two retrievers largely disagree, which is why fusing them helps.


## Why RRF rather than a weighted score sum

Dense cosine is bounded; BM25 is not. Their raw scales are not
comparable, so a weighted sum would compare incommensurable units.

In [5]:
print(f'dense  cosine range: {min(s for _, s in dense):.3f} .. {max(s for _, s in dense):.3f}')
print(f'BM25   score range: {min(s for _, s in sparse):.3f} .. {max(s for _, s in sparse):.3f}')

dense  cosine range: 0.551 .. 0.576
BM25   score range: 14.482 .. 18.810


In [6]:
from branched_rag.index.fusion import reciprocal_rank_fusion

fused = reciprocal_rank_fusion(dense, sparse, 0.5, 0.5, k=60, top_k=5)
for hit in fused:
    print(f'{hit.score:.5f}  dense_rank={hit.dense_rank}  '
          f'lex_rank={hit.sparse_rank}  both={hit.found_by_both}')

0.00820  dense_rank=None  lex_rank=1  both=False
0.00820  dense_rank=1  lex_rank=None  both=False
0.00806  dense_rank=None  lex_rank=2  both=False
0.00806  dense_rank=2  lex_rank=None  both=False
0.00794  dense_rank=None  lex_rank=3  both=False


Passages found by both retrievers rise to the top without their scores ever being compared.

## Full ablation: dense vs sparse vs fused vs fused+rerank

In [7]:
table = evaluate_retrieval(pipeline, sample_size=60, k=10)
table

,branch,mode,queries,recall@10,mrr,ndcg@10
0,reviews,dense,56,0.296,0.299,0.240
1,reviews,sparse,56,0.312,0.341,0.274
2,reviews,fused,56,0.341,0.320,0.274
3,reviews,fused+rerank,56,0.341,0.385,0.313
4,news,dense,60,0.750,0.471,0.539
5,news,sparse,60,0.667,0.377,0.446
6,news,fused,60,0.833,0.475,0.560
7,news,fused+rerank,60,0.867,0.568,0.640
8,financials,dense,60,0.698,0.918,0.730
9,financials,sparse,60,0.667,0.904,0.689


In [8]:
pivot = table.pivot(index='mode', columns='branch', values='ndcg@10')
pivot = pivot.reindex(['dense', 'sparse', 'fused', 'fused+rerank'])
print('nDCG@10 by retrieval mode')
print(pivot.to_string())

nDCG@10 by retrieval mode
branch        financials   news  reviews
mode                                    
dense              0.730  0.539    0.240
sparse             0.689  0.446    0.274
fused              0.740  0.560    0.274
fused+rerank       0.754  0.640    0.313


`fused+rerank` is best in every branch. The `reviews` branch scores lowest
overall because review headlines such as "Disappointed" genuinely do not
identify their document, so part of what the metric measures there is
title quality. The relative comparison across modes remains valid, which
is what it is used for.

## Branch retrieval end to end

In [9]:
answer = pipeline.answer('What do customers complain about in chargers?', generate=False)
print('routed to:', answer.plan.selected)
for result in answer.results:
    print(f'\n{result.label}: query sent = {result.query.text!r}')
    for item in result.evidence[:3]:
        print(f'  [{item.citation}] via {item.retrieval_path}: {item.text[:85]}')

routed to: ['reviews']

Customer Reviews: query sent = 'What do customers complain about in chargers? complaint customer battery'
  [1] via dense + lexical: The only real complaint is the lifespan. In both of my units, the same problem occure
  [2] via dense + lexical: If I had one complaint it would be that the cables are very short, making it somewhat
  [3] via dense + lexical: My only complaint (other than wishing it was cordless) is that it eats batteries. It 
